# Technologies.csv Generation — Reality Scenario — Norte Amazónica 2025

Generates one `Technologies.csv` per cluster (C1–C5) for EnergyScope ESMC, **reality scenario**.
Base: sufficiency output (`../sufficiency/output_energyscope/C{k}/Technologies.csv`).
Only values documented below are overwritten; all other rows are inherited unchanged from sufficiency.

**Rules applied:**
- **(a) Electricity generators** — `f_max = f_min` (locks capacity to existing installed amounts; no new construction)
- **(b) Off-grid techs** — `PV_HS`, `HS_DIESEL`, `BATT_HS` set to `f_min = f_max` = recensed fleet from `source_B_home_systems_reality.csv`
- **(c) Storage** — `f_max = f_min` for all other battery/storage technologies (no new storage)
- **(d) Stoves** — `f_min` recalculated from Census 2024 cooking fuel data; `f_max` unchanged from sufficiency
- **(e) HS_DIESEL production** — `f_min_prod = f_max_prod` = residual Source B demand not met by PV_HS

In [1]:
import os
import pandas as pd

OUT_DIR  = "output_energyscope"
SUFF_DIR = "../sufficiency/output_energyscope"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# Load sufficiency output as base — only modified rows are overwritten
suff = {}
for k in range(1, 6):
    path = os.path.join(SUFF_DIR, f"C{k}", "Technologies.csv")
    df = pd.read_csv(path, sep=";")
    df["Technologies param"] = df["Technologies param"].str.strip()
    suff[k] = df
print("Loaded sufficiency base for C1–C5")

Loaded sufficiency base for C1–C5


## 1. Electricity generation — rule (a)

`f_max = f_min` for all technologies with positive `ELECTRICITY` output in `Layers_in_out.csv`.
This locks each cluster to its existing installed capacity (from AETN 2024 / Census data already encoded in the sufficiency base).
If `f_min = 0`, then `f_max = 0` — no new construction of any generator type.

In [2]:
# Source: Layers_in_out.csv — techs with positive ELECTRICITY coefficient are generators
lio = pd.read_csv("../data/Layers_in_out.csv", sep=";")
lio.columns = [c.strip() for c in lio.columns]
tech_col = lio.columns[0]

ELECTRICITY_GENERATORS = set(
    lio.loc[(lio["ELECTRICITY"] > 0) & (lio[tech_col] != "ELECTRICITY"), tech_col].tolist()
)
print(f"Identified {len(ELECTRICITY_GENERATORS)} electricity generators:")
print(sorted(ELECTRICITY_GENERATORS))

Identified 54 electricity generators:
['BFB_ST_BIOMASS', 'BIOMASS_TO_DIESEL', 'BIOMASS_TO_GASOLINE', 'BIOMASS_TO_JET_FUEL', 'BIOMASS_TO_LFO', 'BIOMASS_TO_METHANOL', 'BIOMASS_TO_POWER', 'BIOWASTE_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_JET_FUEL', 'BIOWASTE_TO_LFO', 'BIOWASTE_TO_METHANOL', 'BIO_HYDROLYSIS', 'CCGT', 'CCGT_AMMONIA', 'CCGT_SUR', 'CFB_ST_BIOMASS', 'COAL_IGCC', 'COAL_US', 'DEC_ADVCOGEN_GAS', 'DEC_ADVCOGEN_H2', 'DEC_COGEN_GAS', 'DEC_COGEN_OIL', 'DHN_COGEN_GAS', 'DHN_COGEN_WASTE', 'DHN_COGEN_WOOD', 'ETHANOL_TO_FUELS', 'FB_ST_BIOMASS', 'FUEL_CELL', 'GENSET_DIESEL', 'GEOTHERMAL', 'HS_DIESEL', 'HYDRO_DAM', 'HYDRO_RIVER', 'IND_COGEN_GAS', 'IND_COGEN_WASTE', 'IND_COGEN_WOOD', 'NUCLEAR', 'NUCLEAR_SMR', 'OCGT', 'PT_POWER_BLOCK', 'PV_HS', 'PV_ROOFTOP', 'PV_UTILITY', 'PYROLYSIS_TO_FUELS', 'PYROLYSIS_TO_LFO', 'ST_BIOMASS', 'ST_POWER_BLOCK', 'ST_SNG', 'TIDAL_RANGE', 'TIDAL_STREAM', 'WAVE', 'WIND_OFFSHORE', 'WIND_ONSHORE']


## 2. Off-grid and storage — rules (b) & (c)

**Rule (b):** `PV_HS`, `HS_DIESEL` and `BATT_HS` → `f_min = f_max` = the recensed installed capacity
from `output_energyscope/source_B_home_systems_reality.csv` (`f_min_PV_HS_GW`, `f_min_HS_DIESEL_GW`,
`f_min_BATT_HS_GWh`), loaded in section 4 below.

This used to lock all three to 0, on the grounds that Source B demand is aggregated at cluster level
so individual off-grid supply techs didn't need their own capacity. That became inconsistent once
rule (e) imposed a non-zero `HS_DIESEL` production floor: a technology can't produce with zero
installed capacity, so `f_max = 0` made the model infeasible. Rules (b) and (e) now describe the same
recensed off-grid fleet — one in capacity, the other in annual production.

**Rule (c):** All other battery/storage technologies → `f_max = f_min`.
No new storage investment in the reality scenario; existing capacity (typically 0 for most techs) is preserved.

In [3]:
# Rule (b): off-grid techs — capacity (f_min = f_max) locked to the recensed fleet,
# read per cluster from source_B_home_systems_reality.csv in section 4 below.
OFF_GRID_FLEET_COL = {
    "PV_HS":     "f_min_PV_HS_GW",
    "HS_DIESEL": "f_min_HS_DIESEL_GW",
    "BATT_HS":   "f_min_BATT_HS_GWh",
}

# Rule (c): storage techs locked (f_max = f_min)
# Includes electrical, thermal, chemical and vehicle storage
STORAGE_TECHS = [
    # Electrical / electrochemical
    "BATT_LI", "CAES", "BEV_BATT", "PHEV_BATT", "DAM_STORAGE", "PHS", "BATT_HS",
    # Thermal
    "TS_DEC_DIRECT_ELEC", "TS_DEC_HP_ELEC", "TS_DEC_THHP_GAS",
    "TS_DEC_COGEN_GAS",   "TS_DEC_COGEN_OIL", "TS_DEC_ADVCOGEN_GAS",
    "TS_DEC_ADVCOGEN_H2", "TS_DEC_BOILER_GAS", "TS_DEC_BOILER_WOOD",
    "TS_DEC_BOILER_OIL",  "TS_DHN_DAILY", "TS_DHN_SEASONAL", "TS_HIGH_TEMP", "TS_COLD",
    # Chemical / other
    "GAS_STORAGE", "H2_STORAGE", "CO2_STORAGE", "AMMONIA_STORAGE",
    "METHANOL_STORAGE", "PT_STORAGE", "ST_STORAGE",
]
print("OFF_GRID_FLEET_COL:", OFF_GRID_FLEET_COL)
print(f"STORAGE_TECHS ({len(STORAGE_TECHS)} entries) defined.")

OFF_GRID_FLEET_COL: {'PV_HS': 'f_min_PV_HS_GW', 'HS_DIESEL': 'f_min_HS_DIESEL_GW', 'BATT_HS': 'f_min_BATT_HS_GWh'}
STORAGE_TECHS (28 entries) defined.


## 3. Cooking stoves — rule (d)

`f_min` for `STOVE_WOOD` and `STOVE_LPG` is derived from Census 2024 cooking fuel data
(file: `CSV_final_in_excel.xlsx`, sheet `data`, rows start at row 4):

| Column (0-indexed) | Field |
|---|---|
| 47 | Leña (wood households) |
| 50 | Gas domiciliario por cañería |
| 51 | Gas en garrafa |

$$\text{wood\_hh} = \text{col}_{47}, \quad \text{lpg\_hh} = \text{col}_{50} + \text{col}_{51}$$

$$f_{\min}^{\text{WOOD}} = \frac{\text{wood\_hh} \times 0.001344023}{0.1875 \times 8760} \;[\text{GW}]$$

$$f_{\min}^{\text{LPG}} = \frac{\text{lpg\_hh} \times 0.001344023}{0.1875 \times 8760} \;[\text{GW}]$$

Where $0.001344023$ GWh/hh/yr is the Census-based useful cooking energy intensity,
and $0.1875$ is the stove capacity factor (`c_p`).

**Disambiguation:** Two municipalities are named *Santa Rosa*.
Department Beni → `Santa_Rosa_Beni` → C1; Department Pando → `Santa_Rosa_Pando` → C4.

In [4]:
# Source: Bolivia Census 2024 — CSV_final_in_excel.xlsx
xl = pd.ExcelFile("../../exctraction of data/output/CSV_final_in_excel.xlsx")
raw = xl.parse(xl.sheet_names[0], header=None)
data = raw.iloc[3:].reset_index(drop=True)  # skip 3 header rows

COOKING_HH = {}  # muni_key → (wood_hh, lpg_hh)
for _, row in data.iterrows():
    muni = str(row[3]).strip() if pd.notna(row[3]) else ""
    dept = str(row[1]).strip() if pd.notna(row[1]) else ""
    if not muni or muni == "nan":
        continue
    wood_hh = int(row[47]) if pd.notna(row[47]) else 0
    gas_dom  = int(row[50]) if pd.notna(row[50]) else 0
    gas_gar  = int(row[51]) if pd.notna(row[51]) else 0
    lpg_hh   = gas_dom + gas_gar
    # Disambiguate the two "Santa Rosa" entries
    if muni == "Santa Rosa" and dept == "Beni":
        key = "Santa_Rosa_Beni"
    elif muni == "Santa Rosa" and dept == "Pando":
        key = "Santa_Rosa_Pando"
    else:
        key = muni.replace(" ", "_")
    COOKING_HH[key] = (wood_hh, lpg_hh)

COOK_INTENSITY = 0.001344023  # GWh per household per year (Census 2024 cooking energy intensity)
CP_STOVE       = 0.1875       # capacity factor for all stoves

stove_wood_fmin = {}
stove_lpg_fmin  = {}
for k, munis in CLUSTERS.items():
    wood_total = sum(COOKING_HH.get(m, (0, 0))[0] for m in munis)
    lpg_total  = sum(COOKING_HH.get(m, (0, 0))[1] for m in munis)
    stove_wood_fmin[k] = (wood_total * COOK_INTENSITY) / (CP_STOVE * 8760)
    stove_lpg_fmin[k]  = (lpg_total  * COOK_INTENSITY) / (CP_STOVE * 8760)

print(f"{'':8} {'wood_hh':>9} {'lpg_hh':>9} {'STOVE_WOOD f_min (GW)':>22} {'STOVE_LPG f_min (GW)':>22}")
for k, munis in CLUSTERS.items():
    wood_total = sum(COOKING_HH.get(m, (0, 0))[0] for m in munis)
    lpg_total  = sum(COOKING_HH.get(m, (0, 0))[1] for m in munis)
    print(f"C{k}       {wood_total:>9} {lpg_total:>9} {stove_wood_fmin[k]:>22.7f} {stove_lpg_fmin[k]:>22.7f}")

           wood_hh    lpg_hh  STOVE_WOOD f_min (GW)   STOVE_LPG f_min (GW)
C1            5017      5656              0.0041053              0.0046282
C2             340       452              0.0002782              0.0003699
C3            7014     32377              0.0057394              0.0264934
C4            5775     10520              0.0047256              0.0086083
C5             300     14696              0.0002455              0.0120254


## 4. HS_DIESEL production floor — rule (e)

`f_min_prod = f_max_prod` for `HS_DIESEL`, fixing its annual production to the residual Source B demand
not met by `PV_HS`:

$$f_{\min\_prod}^{\text{HS\_DIESEL}} = f_{\max\_prod}^{\text{HS\_DIESEL}} = E_{\text{Source B}} - F_{\min}^{\text{PV\_HS}} \times \sum_h \text{PV}_h$$

Where $E_{\text{Source B}}$ and $F_{\min}^{\text{PV\_HS}}$ (installed PV capacity, GW) come from
`output_energyscope/source_B_home_systems_reality.csv` (produced by `home_systems.ipynb`), and
$\sum_h \text{PV}_h$ is the cluster's annual sum of the `PV` capacity-factor profile in
`output_energyscope/C{k}/Time_series.csv` (produced by `time_series.ipynb`).

All other technologies keep the structural defaults `f_min_prod = 0`, `f_max_prod = 1e15`
(inactive — see `esmc/energy_model/ESMC_model_AMPL.mod`, `f_min_prod_abs`/`f_max_prod_abs`).

In [5]:
source_b = pd.read_csv("output_energyscope/source_B_home_systems_reality.csv", index_col="cluster")

hs_diesel_prod = {}
for k in range(1, 6):
    ts = pd.read_csv(os.path.join(OUT_DIR, f"C{k}", "Time_series.csv"), sep=";")
    pv_annual_sum = ts["PV"].sum()
    pv_hs_prod = source_b.loc[k, "f_min_PV_HS_GW"] * pv_annual_sum
    hs_diesel_prod[k] = source_b.loc[k, "sourceB_elec_demand_GWh"] - pv_hs_prod

print(f"{'':8} {'PV_HS prod (GWh)':>18} {'HS_DIESEL prod (GWh)':>22}")
for k in range(1, 6):
    ts = pd.read_csv(os.path.join(OUT_DIR, f"C{k}", "Time_series.csv"), sep=";")
    pv_hs_prod = source_b.loc[k, "f_min_PV_HS_GW"] * ts["PV"].sum()
    print(f"C{k}       {pv_hs_prod:>18.6f} {hs_diesel_prod[k]:>22.6f}")

           PV_HS prod (GWh)   HS_DIESEL prod (GWh)
C1                 0.131678               0.659460
C2                 0.016817               0.086417
C3                 0.106072               0.688174
C4                 0.102239               1.044259
C5                 0.013471               0.108510


## 5. Assemble and save

In [6]:
for k in range(1, 6):
    df = suff[k].copy()

    # Rule (a): Lock all electricity generation — f_max = f_min
    mask_gen = df["Technologies param"].isin(ELECTRICITY_GENERATORS)
    df.loc[mask_gen, "f_max"] = df.loc[mask_gen, "f_min"]

    # Rule (b): off-grid techs — capacity locked to the recensed fleet (f_min = f_max)
    for tech, col in OFF_GRID_FLEET_COL.items():
        mask = df["Technologies param"] == tech
        df.loc[mask, "f_min"] = source_b.loc[k, col]
        df.loc[mask, "f_max"] = source_b.loc[k, col]

    # Rule (c): Lock storage — f_max = f_min (no new storage investment)
    mask_stor = df["Technologies param"].isin(STORAGE_TECHS)
    df.loc[mask_stor, "f_max"] = df.loc[mask_stor, "f_min"]

    # Disable ST_SNG in reality scenario: f_min = f_max = 0 for all clusters
    df.loc[df["Technologies param"] == "ST_SNG", "f_min"] = 0.0
    df.loc[df["Technologies param"] == "ST_SNG", "f_max"] = 0.0

    # LED-only lighting: conventional bulb/tube techs disabled (f_max = 0)
    df.loc[df["Technologies param"] == "CONVENTIONAL_BULB",  "f_max"] = 0.0
    df.loc[df["Technologies param"] == "CONVENTIONAL_LIGHT", "f_max"] = 0.0

    # Rule (d): Stove f_min from Census 2024 — f_max unchanged from sufficiency
    df.loc[df["Technologies param"] == "STOVE_WOOD", "f_min"] = stove_wood_fmin[k]
    df.loc[df["Technologies param"] == "STOVE_LPG",  "f_min"] = stove_lpg_fmin[k]

    # Rule (e): HS_DIESEL annual production floor/ceiling — residual Source B demand
    df.loc[df["Technologies param"] == "HS_DIESEL", "f_min_prod"] = hs_diesel_prod[k]
    df.loc[df["Technologies param"] == "HS_DIESEL", "f_max_prod"] = hs_diesel_prod[k]

    out_path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_csv(out_path, sep=";", index=False)

print("Saved Technologies.csv for C1–C5")

Saved Technologies.csv for C1–C5


## 6. Verification

In [7]:
clusters_out = {}
for k in range(1, 6):
    path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    df = pd.read_csv(path, sep=";")
    df["Technologies param"] = df["Technologies param"].str.strip()
    clusters_out[k] = df

def lookup(df, tech, col):
    row = df.loc[df["Technologies param"] == tech, col]
    return float(row.values[0]) if len(row) else float("nan")

header = f"{'Technology':<32}" + "".join(f"  C{k:>11}" for k in range(1, 6))
sep    = "-" * len(header)

# --- Rule (a): generators locked ---
print("=== Rule (a): generators locked (f_max == f_min) ===")
print(header); print(sep)
for tech in ["GENSET_DIESEL", "PV_UTILITY"]:
    for col in ["f_min", "f_max"]:
        vals = [lookup(clusters_out[k], tech, col) for k in range(1, 6)]
        print(f"{tech+' '+col:<32}" + "".join(f"  {v:>11.5f}" for v in vals))
    locked = ["OK" if abs(lookup(clusters_out[k], tech, "f_min") -
                         lookup(clusters_out[k], tech, "f_max")) < 1e-9
              else "FAIL" for k in range(1, 6)]
    print(f"{'  locked?':<32}" + "".join(f"  {s:>11}" for s in locked))

print()

# --- Rule (b): off-grid fleet locked to source_B_home_systems_reality.csv ---
print("=== Rule (b): off-grid capacity == recensed fleet (f_min == f_max == source_b) ===")
print(header); print(sep)
for tech, col in OFF_GRID_FLEET_COL.items():
    for c in ["f_min", "f_max"]:
        vals = [lookup(clusters_out[k], tech, c) for k in range(1, 6)]
        print(f"{tech+' '+c:<32}" + "".join(f"  {v:>11.7f}" for v in vals))
    matches = ["OK" if abs(lookup(clusters_out[k], tech, "f_min") - source_b.loc[k, col]) < 1e-9 and
                       abs(lookup(clusters_out[k], tech, "f_max") - source_b.loc[k, col]) < 1e-9
               else "FAIL" for k in range(1, 6)]
    print(f"{'  == source_b?':<32}" + "".join(f"  {s:>11}" for s in matches))

print()

# --- Rule (c): storage locked ---
print("=== Rule (c): storage locked (f_max == f_min) ===")
print(header); print(sep)
for tech in ["BATT_LI", "BATT_HS"]:
    for col in ["f_min", "f_max"]:
        vals = [lookup(clusters_out[k], tech, col) for k in range(1, 6)]
        print(f"{tech+' '+col:<32}" + "".join(f"  {v:>11.5f}" for v in vals))

print()

# --- LED-only lighting ---
print("=== LED-only lighting (CONVENTIONAL_BULB / CONVENTIONAL_LIGHT f_max == 0) ===")
print(header); print(sep)
for tech in ["CONVENTIONAL_BULB", "CONVENTIONAL_LIGHT"]:
    vals = [lookup(clusters_out[k], tech, "f_max") for k in range(1, 6)]
    ok   = ["OK" if v == 0.0 else "FAIL" for v in vals]
    print(f"{tech+' f_max':<32}" + "".join(f"  {v:>11.5f}" for v in vals))
    print(f"{'  disabled?':<32}" + "".join(f"  {s:>11}" for s in ok))

print()

# --- Rule (d): stove f_min + cooking demand check ---
print("=== Rule (d): stove f_min and cooking demand check ===")
print(header); print(sep)
for tech in ["STOVE_WOOD", "STOVE_LPG"]:
    for col in ["f_min", "f_max"]:
        vals = [lookup(clusters_out[k], tech, col) for k in range(1, 6)]
        print(f"{tech+' '+col:<32}" + "".join(f"  {v:>11.5f}" for v in vals))

print()
print("Cooking demand check -- (f_min_WOOD + f_min_LPG) * 0.1875 * 8760 <= COOKING [GWh]:")
for k in range(1, 6):
    d = pd.read_csv(os.path.join(OUT_DIR, f"C{k}", "Demands.csv"), sep=";")
    cooking_row = d[d["parameter name"] == "COOKING"]
    cook_demand = float(
        cooking_row.drop(columns=["Category", "Subcategory", "parameter name", "Units"],
                         errors="ignore").sum(axis=1).iloc[0]
    )
    wood_fmin = lookup(clusters_out[k], "STOVE_WOOD", "f_min")
    lpg_fmin  = lookup(clusters_out[k], "STOVE_LPG",  "f_min")
    stove_contrib = (wood_fmin + lpg_fmin) * CP_STOVE * 8760
    ok = "OK" if stove_contrib <= cook_demand + 1e-6 else "EXCEEDS DEMAND"
    print(f"  C{k}: stove_contrib = {stove_contrib:.4f} GWh  |  demand = {cook_demand:.4f} GWh  [{ok}]")

print()

# --- Rule (e): HS_DIESEL production floor/ceiling ---
print("=== Rule (e): HS_DIESEL f_min_prod == f_max_prod == residual Source B demand ===")
print(header); print(sep)
for col in ["f_min_prod", "f_max_prod"]:
    vals = [lookup(clusters_out[k], "HS_DIESEL", col) for k in range(1, 6)]
    print(f"{'HS_DIESEL '+col:<32}" + "".join(f"  {v:>11.5f}" for v in vals))
locked = ["OK" if abs(lookup(clusters_out[k], "HS_DIESEL", "f_min_prod") -
                     lookup(clusters_out[k], "HS_DIESEL", "f_max_prod")) < 1e-9
          else "FAIL" for k in range(1, 6)]
print(f"{'  locked?':<32}" + "".join(f"  {s:>11}" for s in locked))

print()

# --- Cross-check: f_min_prod > 0 implies f_max > 0 (else infeasible) ---
print("=== Cross-check: no tech has f_min_prod > 0 with f_max == 0 ===")
for k in range(1, 6):
    bad = clusters_out[k][(clusters_out[k]["f_min_prod"] > 0) & (clusters_out[k]["f_max"] <= 0)]
    status = "OK" if bad.empty else f"FAIL: {bad['Technologies param'].tolist()}"
    print(f"  C{k}: {status}")

=== Rule (a): generators locked (f_max == f_min) ===
Technology                        C          1  C          2  C          3  C          4  C          5
------------------------------------------------------------------------------------------------------
GENSET_DIESEL f_min                   0.00000      0.00000      0.05333      0.00742      0.02736
GENSET_DIESEL f_max                   0.00000      0.00000      0.05333      0.00742      0.02736
  locked?                                  OK           OK           OK           OK           OK
PV_UTILITY f_min                      0.00000      0.00000      0.00000      0.00000      0.00510
PV_UTILITY f_max                      0.00000      0.00000      0.00000      0.00000      0.00510
  locked?                                  OK           OK           OK           OK           OK

=== Rule (b): off-grid capacity == recensed fleet (f_min == f_max == source_b) ===
Technology                        C          1  C          2  C      

BATT_LI f_min                         0.00000      0.00000      0.00000      0.00000      0.00000
BATT_LI f_max                         0.00000      0.00000      0.00000      0.00000      0.00000
BATT_HS f_min                         0.00021      0.00003      0.00017      0.00017      0.00002
BATT_HS f_max                         0.00021      0.00003      0.00017      0.00017      0.00002

=== LED-only lighting (CONVENTIONAL_BULB / CONVENTIONAL_LIGHT f_max == 0) ===
Technology                        C          1  C          2  C          3  C          4  C          5
------------------------------------------------------------------------------------------------------
CONVENTIONAL_BULB f_max               0.00000      0.00000      0.00000      0.00000      0.00000
  disabled?                                OK           OK           OK           OK           OK
CONVENTIONAL_LIGHT f_max              0.00000      0.00000      0.00000      0.00000      0.00000
  disabled?                  

  C2: stove_contrib = 1.0645 GWh  |  demand = 1.0712 GWh  [OK]
  C3: stove_contrib = 52.9424 GWh  |  demand = 54.1810 GWh  [OK]
  C4: stove_contrib = 21.9009 GWh  |  demand = 22.2733 GWh  [OK]
  C5: stove_contrib = 20.1550 GWh  |  demand = 20.9590 GWh  [OK]

=== Rule (e): HS_DIESEL f_min_prod == f_max_prod == residual Source B demand ===
Technology                        C          1  C          2  C          3  C          4  C          5
------------------------------------------------------------------------------------------------------
HS_DIESEL f_min_prod                  0.65946      0.08642      0.68817      1.04426      0.10851
HS_DIESEL f_max_prod                  0.65946      0.08642      0.68817      1.04426      0.10851
  locked?                                  OK           OK           OK           OK           OK

=== Cross-check: no tech has f_min_prod > 0 with f_max == 0 ===
  C1: OK
  C2: OK
  C3: OK
  C4: OK
  C5: OK
